Food type and freshness. Runtime: L4.

Pull `ftaf_dev`, unzip, install, then `prep`. The next cell keeps `last.pt` when it is already there. `fresh` audits 30 epochs and refits only if the gate passes. If it prints `stop`, do not predict and do not submit.


In [ ]:
import os
from pathlib import Path

repo = Path("/content/fve_bosch")
if not (repo / "baselines" / "fresh.py").is_file():
    !git clone -b ftaf_dev --depth 1 https://github.com/TinhAnhGitHub/fve_bosch.git {repo}
else:
    !git -C {repo} pull --ff-only
os.chdir(repo)
print(repo)

In [ ]:
import glob, os, zipfile
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

src = glob.glob("/content/drive/MyDrive/ftaf*.zip")[0]
print(src, os.path.getsize(src), flush=True)
dest = Path("/content/fve_bosch/data/ftaf")
dest.mkdir(parents=True, exist_ok=True)
if not any(dest.rglob("annotations.json")):
    with zipfile.ZipFile(src) as z:
        z.extractall(dest)
print("annotations", list(dest.rglob("annotations.json")))

In [ ]:
!pip -q install ultralytics
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
!python baselines/fresh.py prep

In [ ]:
from pathlib import Path
import subprocess, sys
det = Path("baselines/out/fresh/det")
weights = list(det.rglob("last.pt")) or list(det.rglob("best.pt"))
if weights:
    print("keep detector", weights[0])
else:
    print("no detector, training once")
    subprocess.check_call([sys.executable, "baselines/fresh.py", "train"])


In [ ]:
# Audits 30 epochs on detector boxes. Holds out a fifth of each food.
# Keeps the epoch near the best kappa with the fewest two-grade misses.
# Retrains that many epochs on every photo only if the epoch is 8..29,
# two-grade misses beat the last epoch, and kappa is at least 0.52.
# If this prints stop, do not run the next cell and do not submit.
!python baselines/fresh.py fresh


In [ ]:
from pathlib import Path
import subprocess, sys
ckpt = Path("baselines/out/fresh/coral.pt")
if not ckpt.is_file():
    raise SystemExit("no coral.pt — fresh printed stop, do not predict")
subprocess.check_call([sys.executable, "baselines/fresh.py", "predict"])


In [ ]:
from google.colab import files
files.download("submissions/fresh/preds.json")